# How CatBoost's `select_features` works

`CatBoostClassifier.select_features` is CatBoost's own recursive feature elimination. This notebook
checks on simulated data what the catboost 1.2.10 source code says it does:

1. how many columns each step removes, and a bug: a step that should remove no column removes all
   the remaining ones (PredictionValuesChange and LossFunctionChange);
2. what the three algorithms (PredictionValuesChange, LossFunctionChange, ShapValues) keep, how
   long they take, and the quality of the final model;
3. which points of `loss_graph` are measured and which are estimated, and how far off the
   estimates are;
4. how optimistic a score on the `eval_set` is after the selection used it;
5. that the columns are scored by `loss_function`, yet `eval_metric` can still change the order of
   removal through the best iteration;
6. that columns derived from an eliminated text column stay in the model;
7. selection by groups of columns (`grouping="ByTags"`), and whether a tag's cost changes it.

The data: 14 useful columns, 8 of them additive and 6 in three interacting pairs (a four-level
`city` that moves a price threshold, an hour that flips the sign of a distance effect, and the XOR
of two Gaussians `u` and `v`), plus 40 Gaussian and 8 four-level noise columns. Every draw uses a
named seed. The run takes about 9 minutes on an Apple M4 Pro with 14 cores; the last cell prints the time.

In [1]:
from importlib.metadata import version, PackageNotFoundError

try:
    catboost_ok = version("catboost") == "1.2.10"
except PackageNotFoundError:
    catboost_ok = False
if not catboost_ok:
    !pip install -q catboost==1.2.10

import os
import time
import warnings
from contextlib import contextmanager

import catboost
import numpy as np
import pandas as pd
import sklearn
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import log_loss, roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200, "display.max_columns", 30)
N_JOBS = int(os.environ.get("N_JOBS", -1))  # CatBoost threads; results do not depend on it
print(f"catboost {catboost.__version__}, numpy {np.__version__}, pandas {pd.__version__}, sklearn {sklearn.__version__}")
TIMES, T0 = {}, time.perf_counter()


@contextmanager
def timed(block):
    t = time.perf_counter()
    yield
    TIMES[block] = time.perf_counter() - t

catboost 1.2.10, numpy 2.5.3, pandas 3.0.5, sklearn 1.9.1


## Data

Training rows, an `eval_set` and a fresh draw of the same size as the training rows. The selection
sees the training rows and the `eval_set`; the fresh draw is only for scoring.

In [2]:
CITY_LEVELS = np.log([300, 600, 900, 1500])
ADD_COLS = [f"add_{i + 1}" for i in range(8)]
PAIR_COLS = ["city", "log_price", "hour", "distance_km", "u", "v"]
USEFUL = ADD_COLS + PAIR_COLS
INTER_ONLY = ["city", "hour", "u", "v"]   # no effect of their own, only through their pair
GAUSS_NOISE = [f"gnoise_{i + 1}" for i in range(40)]
CAT_NOISE = [f"cnoise_{i + 1}" for i in range(8)]
CATS = ["city"] + CAT_NOISE
K = len(USEFUL)


def gen(seed, n):
    rng = np.random.default_rng(seed)
    add = rng.normal(size=(n, 8))
    city = rng.integers(0, 4, n)
    log_price = CITY_LEVELS[city] + rng.normal(0, 0.25, n)
    hour = rng.uniform(0, 24, n)
    distance_km = rng.lognormal(mean=np.log(6), sigma=0.6, size=n)
    z = (np.log(distance_km) - np.log(6)) / 0.6
    u, v = rng.normal(size=n), rng.normal(size=n)
    logit = (add @ (0.4 * np.array([1, -1] * 4)) + 8 * (log_price - CITY_LEVELS[city]) + 0.3 * z
             + 2.5 * np.cos(2 * np.pi * hour / 24) * z + 1.5 * np.sign(u) * np.sign(v) - 1.0)
    y = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    df = pd.DataFrame(add, columns=ADD_COLS).assign(city=city, log_price=log_price, hour=hour,
                                                      distance_km=distance_km, u=u, v=v)
    df = pd.concat([df, pd.DataFrame(rng.normal(size=(n, 40)), columns=GAUSS_NOISE),
                    pd.DataFrame(rng.integers(0, 4, (n, 8)), columns=CAT_NOISE)], axis=1)
    return df, y


X, y = gen(1, 10_000)
X_eval, y_eval = gen(2, 5_000)
X_fresh, y_fresh = gen(3, 10_000)
COLS = list(X.columns)
print(f"{len(COLS)} columns ({K} useful), base rate {y.mean():.3f}")


def model(**kw):
    params = dict(iterations=300, depth=6, learning_rate=0.1, random_seed=0, verbose=0, thread_count=N_JOBS,
                  allow_writing_files=False, cat_features=CATS)
    params.update(kw)
    return CatBoostClassifier(**params)


def select(algorithm, steps, final=True, cols=COLS, k=K, **kw):
    # one select_features call; returns the fitted (or unfitted) model, the result dict and seconds
    m, t = model(**kw), time.perf_counter()
    r = m.select_features(X, y, eval_set=(X_eval, y_eval), features_for_select=cols, num_features_to_select=k,
                          steps=steps, algorithm=algorithm, train_final_model=final, logging_level="Silent")
    return m, r, time.perf_counter() - t


GROUPS = {"useful": USEFUL, "interaction-only": INTER_ONLY, "Gaussian noise": GAUSS_NOISE, "four-level noise": CAT_NOISE}


def counts(kept):
    return {g: sum(c in cols for c in kept) for g, cols in GROUPS.items()}

62 columns (14 useful), base rate 0.390


## The call and what comes back

`num_features_to_select` and `features_for_select` are required (for groups of columns, their tag
versions replace them; see the last section). The defaults are
`algorithm="RecursiveByShapValues"`, `steps=1` and `train_final_model=True`: after the selection the
estimator is refit on the kept columns and overwritten. `eval_set` is optional; without it the
columns are scored on the training rows.

In [3]:
with timed("call"):
    m_shap, r_shap, sec_shap = select("RecursiveByShapValues", 5)
print("keys:", list(r_shap))
print("loss_graph keys:", list(r_shap["loss_graph"]))
print("first 10 removed, in order:", r_shap["eliminated_features_names"][:10])
print("kept:", sorted(r_shap["selected_features_names"]))
print(counts(r_shap["selected_features_names"]), f"{sec_shap:.0f} s")

keys: ['selected_features', 'eliminated_features_names', 'loss_graph', 'eliminated_features', 'selected_features_names']
loss_graph keys: ['main_indices', 'removed_features_count', 'loss_values']
first 10 removed, in order: ['gnoise_18', 'gnoise_4', 'gnoise_32', 'gnoise_6', 'gnoise_23', 'gnoise_16', 'gnoise_2', 'gnoise_3', 'gnoise_26', 'gnoise_17']
kept: ['add_1', 'add_2', 'add_3', 'add_4', 'add_5', 'add_6', 'add_7', 'add_8', 'city', 'distance_km', 'hour', 'log_price', 'u', 'v']
{'useful': 14, 'interaction-only': 4, 'Gaussian noise': 0, 'four-level noise': 0} 37 s


## How many columns each step removes

The model is refit once per step, not once per removed column. The number removed per step shrinks
geometrically: with p = (k / n)^(1 / steps), step s = 0, 1, ... removes about n p^s (1 - p),
rounded half up, with the rounding carried to the next step. The next cell prints `measured`, the cumulative number of removed columns
at each measured point of `loss_graph`, next to the formula's count per step and its cumulative
count before each step, and `kept`, the length of the kept list. With `train_final_model=False`
there is no measured point after the last step for LFC (used here) and ShapValues; PVC fits once
more for that point (source line 825). The last two rows run 5 columns down to 1 in 4 steps and 10
columns down to 1 in 9 steps, where the formula has a step that removes nothing.

In [4]:
ALGOS = {"PVC": "RecursiveByPredictionValuesChange", "LFC": "RecursiveByLossFunctionChange",
         "SHAP": "RecursiveByShapValues"}
ALGO_NAMES = list(ALGOS.values())


def formula(n, k, steps):
    p, carry, out = (k / n) ** (1 / steps), 0.0, []
    for s in range(steps):
        x = n * p ** s * (1 - p) + carry
        out.append(int(np.floor(x + 0.5)))   # C++ round: half away from zero
        carry = x - out[-1]
    return out


rows = []
with timed("schedule"):
    for cols, k, steps in ((COLS, K, 1), (COLS, K, 3), (COLS, K, 10), (COLS[:5], 1, 4), (COLS[:10], 1, 9)):
        _, r, _ = select("RecursiveByLossFunctionChange", steps, final=False, cols=cols, k=k)
        g, f = r["loss_graph"], formula(len(cols), k, steps)
        rows.append({"columns": f"{len(cols)} -> {k}" + (" (the other columns kept)" if cols != COLS else ""),
                     "steps": steps, "measured": [g["removed_features_count"][i] for i in g["main_indices"]],
                     # removed before each step: the measured points with train_final_model=False
                     "formula, cumulative": np.cumsum([0] + f[:-1]).tolist(), "formula per step": f,
                     "kept": len(r["selected_features_names"])})
with pd.option_context("display.max_colwidth", 80):
    print(pd.DataFrame(rows).to_string(index=False))

                         columns  steps                               measured                    formula, cumulative               formula per step  kept
                        62 -> 14      1                                    [0]                                    [0]                           [48]    14
                        62 -> 14      3                            [0, 24, 39]                            [0, 24, 39]                    [24, 15, 9]    14
                        62 -> 14     10 [0, 9, 16, 22, 28, 33, 37, 40, 43, 46] [0, 9, 16, 22, 28, 33, 37, 40, 43, 46] [9, 7, 6, 6, 5, 4, 3, 3, 3, 2]    14
 5 -> 1 (the other columns kept)      4                           [0, 2, 3, 4]                           [0, 2, 3, 4]                   [2, 1, 1, 0]     0
10 -> 1 (the other columns kept)      9          [0, 2, 4, 5, 6, 7, 8, 10, 10]            [0, 2, 4, 5, 6, 7, 8, 8, 9]    [2, 2, 1, 1, 1, 1, 0, 1, 0]     0


In the first three rows the measured points match the formula. In the last row the formula's 7th
step removes nothing, yet the measured count jumps from 8 to 10: all 10 columns are removed and the
kept list is empty, although `num_features_to_select=1`. The 5 -> 1 row, one step per removed column,
also has a zero step in the formula and keeps no column either. The cause is in
`catboost/libs/features_selection/recursive_features_elimination.cpp` (lines 519 and 565 at tag
v1.2.10): PVC and LFC stop removing when `++eliminatedEntitiesCount == numEntitiesToEliminateAtThisStep`,
which never holds when the step's count is 0, so the loop removes every remaining column. ShapValues
counts with `i < n` and is not affected. If every column were in `features_for_select`, the next
fit would have no columns; that case is not run here. The next cell runs 10 columns down to `k` on small data for every
number of steps.

In [5]:
rng = np.random.default_rng(5)
Xs = pd.DataFrame(rng.normal(size=(3_000, 11)), columns=[f"x{i}" for i in range(11)])
ys = (Xs.x0 + Xs.x1 + rng.normal(size=3_000) > 0).astype(int)
rows = []
for algorithm in ALGO_NAMES:
    for k in (1, 2, 3):
        kept = []
        for steps in range(1, 11 - k):
            # x10 stays outside the selection, so every fit has at least one column
            r = model(iterations=30, cat_features=None).select_features(
                Xs[:2_000], ys[:2_000], eval_set=(Xs[2_000:], ys[2_000:]), features_for_select=list(Xs.columns[:10]),
                num_features_to_select=k, steps=steps, algorithm=algorithm, train_final_model=False,
                logging_level="Silent")
            kept.append(len(r["selected_features_names"]))
        zero = [s for s in range(1, 11 - k) if 0 in formula(10, k, s)]
        rows.append({"algorithm": algorithm, "k": k, "columns kept, steps = 1, 2, ...": kept,
                     "steps with a zero step in the formula": zero})
with pd.option_context("display.max_colwidth", 60):
    print(pd.DataFrame(rows).to_string(index=False))

                        algorithm  k columns kept, steps = 1, 2, ... steps with a zero step in the formula
RecursiveByPredictionValuesChange  1     [1, 1, 1, 1, 1, 0, 0, 0, 0]                          [6, 7, 8, 9]
RecursiveByPredictionValuesChange  2        [2, 2, 2, 2, 2, 0, 0, 0]                             [6, 7, 8]
RecursiveByPredictionValuesChange  3           [3, 3, 3, 3, 3, 0, 0]                                [6, 7]
    RecursiveByLossFunctionChange  1     [1, 1, 1, 1, 1, 0, 0, 0, 0]                          [6, 7, 8, 9]
    RecursiveByLossFunctionChange  2        [2, 2, 2, 2, 2, 0, 0, 0]                             [6, 7, 8]
    RecursiveByLossFunctionChange  3           [3, 3, 3, 3, 3, 0, 0]                                [6, 7]
            RecursiveByShapValues  1     [1, 1, 1, 1, 1, 1, 1, 1, 1]                          [6, 7, 8, 9]
            RecursiveByShapValues  2        [2, 2, 2, 2, 2, 2, 2, 2]                             [6, 7, 8]
            RecursiveByShapValues  3 

## Three algorithms

PredictionValuesChange (PVC) scores the columns from the trees alone; LossFunctionChange (LFC) and
ShapValues score them on the `eval_set`. All three refit once per step, on the training rows with
the `eval_set` picking the best iteration. PVC and LFC remove the step's columns at once, by one
importance vector. ShapValues removes them one at a time inside the step: it subtracts each remaining column's SHAP values from
the current prediction, removes the column whose subtraction raises the loss least, keeps that
subtraction, and repeats, without refitting. Same target count and 5 steps for all three; the
references are CatBoost on all columns and on the 14 useful ones.

In [6]:
def scores(m):
    # log loss on the eval_set (which the selection saw) and on the fresh draw, and the fresh AUC
    cols = m.feature_names_
    pe, pf = m.predict_proba(X_eval[cols])[:, 1], m.predict_proba(X_fresh[cols])[:, 1]
    return {"eval log loss": log_loss(y_eval, pe), "fresh log loss": log_loss(y_fresh, pf),
            "fresh AUC": roc_auc_score(y_fresh, pf)}


res, kept_by = [], {}
with timed("algorithms"):
    for name, algorithm in ALGOS.items():
        m, r, sec = (m_shap, r_shap, sec_shap) if name == "SHAP" else select(algorithm, 5)
        kept_by[name] = sorted(r["selected_features_names"])
        res.append(dict(method=name, seconds=round(sec), **counts(kept_by[name]), **scores(m)))
    for name, kept in (("all columns", COLS), (f"the {K} useful", USEFUL)):
        m = model(cat_features=[c for c in CATS if c in kept]).fit(X[kept], y, eval_set=(X_eval[kept], y_eval))
        res.append(dict(method=name, seconds=np.nan, **counts(kept), **scores(m)))
RES = pd.DataFrame(res).set_index("method")
RES["optimism"] = RES["fresh log loss"] - RES["eval log loss"]
print(RES.round(4).to_string())

               seconds  useful  interaction-only  Gaussian noise  four-level noise  eval log loss  fresh log loss  fresh AUC  optimism
method                                                                                                                                
PVC               34.0      14                 4               0                 0         0.3701          0.3588     0.9182   -0.0113
LFC               36.0      14                 4               0                 0         0.3701          0.3588     0.9182   -0.0113
SHAP              37.0      14                 4               0                 0         0.3701          0.3588     0.9182   -0.0113
all columns        NaN      14                 4              40                 8         0.3975          0.3861     0.9107   -0.0114
the 14 useful      NaN      14                 4               0                 0         0.3688          0.3579     0.9186   -0.0109


`optimism` is the fresh log loss minus the eval log loss. In the selection rows the `eval_set`
picks the best iteration of every fit, including the final one, and for LFC and ShapValues it also
scores the columns. The reference rows use it for the best iteration only. All rows also carry the
difference between two random draws (5,000 eval rows, 10,000 fresh rows), which can go either way,
so compare the selection rows with the reference rows, not with zero.

All three algorithms keep exactly the 14 useful columns, so their final models are the same, and
each takes 34 to 37 seconds with 5 steps. The reference row on the same 14 columns differs in the
fourth decimal: a plain fit is not the selection's fit (see the next section). `optimism` is about
-0.011 in every row, the reference rows included: this eval draw is harder than the fresh one, and the selection adds no optimism
visible at this scale, since it picked the true columns. With weaker signal the `eval_set` has more
room to pick columns that fit it by chance; that is not tested here.

## `loss_graph`: measured and estimated points

For LFC and ShapValues `loss_graph` has a loss value after every removed column (PVC has only the
measured ones). Only the points at `main_indices` come from a refit; the points between are estimates (for ShapValues, the loss with the removed columns'
SHAP values subtracted), rescaled so the segment ends at the next measured value. The next cell
gets the true loss at a point by calling `select_features` again with one step, the removed columns
passed as `ignored_features` (by index), and reading its first `loss_graph` value, which comes from
a refit. A plain `fit` with the same columns is not the same fit: with all columns it gives 0.3975 on the
`eval_set` (the "all columns" row above) against 0.3939 for the first point of the graph.
It does this at three measured points, as a check, and at several points between them.

In [7]:
g, elim = r_shap["loss_graph"], r_shap["eliminated_features_names"]
main = sorted(set(g["main_indices"]))
between = [i for i in range(1, len(g["loss_values"]) - 1) if i not in main]
picks = main[:3] + between[:: max(1, len(between) // 6)][:6]
rows = []
with timed("loss graph"):
    for i in picks:
        removed = g["removed_features_count"][i]
        rest = [c for c in COLS if c not in elim[:removed]]
        rr = model(ignored_features=[COLS.index(c) for c in elim[:removed]]).select_features(
            X, y, eval_set=(X_eval, y_eval), features_for_select=rest, num_features_to_select=len(rest) - 1,
            steps=1, algorithm="RecursiveByShapValues", train_final_model=False, logging_level="Silent")
        rows.append(dict(removed=removed, measured=i in main, graph=g["loss_values"][i],
                         refit=rr["loss_graph"]["loss_values"][0]))
LG = pd.DataFrame(rows)
LG["refit - graph"] = LG.refit - LG.graph
print("main_indices (measured):", main)
print(LG.round(4).to_string(index=False))

main_indices (measured): [0, 16, 28, 37, 43, 48]
 removed  measured  graph  refit  refit - graph
       0      True 0.3939 0.3939         0.0000
      16      True 0.3844 0.3844         0.0000
      28      True 0.3738 0.3738         0.0000
       1     False 0.3929 0.4002         0.0073
       8     False 0.3881 0.3842        -0.0039
      15     False 0.3848 0.3819        -0.0029
      23     False 0.3766 0.3782         0.0016
      31     False 0.3702 0.3734         0.0032
      39     False 0.3692 0.3685        -0.0006


At the measured points this refit matches the graph exactly. The estimated points are off by up to
about 0.007 in log loss in either direction, so they show the shape of the curve, not the exact
loss after each removed column.

## `eval_metric` and the best iteration

The columns are scored by `loss_function` (Logloss here), and `loss_graph` holds that loss. With an
`eval_set`, CatBoost also keeps the best iteration of each fit by `eval_metric`, so a different
`eval_metric` gives different models and can change the selection. The next cell repeats the SHAP
selection with `eval_metric="AUC"`, then both versions with `use_best_model=False`.

In [8]:
def same(a, b):
    return a["eliminated_features_names"] == b["eliminated_features_names"]


with timed("eval_metric"):
    _, r_auc, _ = select("RecursiveByShapValues", 5, final=False, eval_metric="AUC")
    _, r_last, _ = select("RecursiveByShapValues", 5, final=False, use_best_model=False)
    _, r_auc_last, _ = select("RecursiveByShapValues", 5, final=False, eval_metric="AUC", use_best_model=False)
print("eval_metric AUC vs default, same removal order:", same(r_auc, r_shap),
      f"; same kept set: {set(r_auc['selected_features_names']) == set(r_shap['selected_features_names'])}")
print("first loss_graph values, default and AUC:", np.round(r_shap["loss_graph"]["loss_values"][:3], 4),
      np.round(r_auc["loss_graph"]["loss_values"][:3], 4))
print("use_best_model=False, eval_metric AUC vs default, same removal order:", same(r_auc_last, r_last))

eval_metric AUC vs default, same removal order: False ; same kept set: True
first loss_graph values, default and AUC: [0.3939 0.3929 0.392 ] [0.3939 0.3929 0.392 ]
use_best_model=False, eval_metric AUC vs default, same removal order: True


With `eval_metric="AUC"` the removal order changes, though here the kept set does not. The first
`loss_graph` values are identical in both runs: they stay Logloss under `eval_metric="AUC"`, and the
first fit stops at the same best iteration under both metrics. With `use_best_model=False` the two
orders are the same, so the change comes from the best iteration.

## An eliminated text column stays in the model

Text columns are turned into derived numeric columns before training. In catboost 1.2.10 removing
the source column does not remove the derived ones (the source has a `TODO` at this spot,
`catboost/private/libs/algo/data.cpp`). Here the text column has no link to the target, the
selection removes it, and the final model is checked for importance on it.

In [9]:
rng = np.random.default_rng(4)
lab = rng.integers(0, 2, 600)
td = pd.DataFrame({"txt": rng.choice(["alpha beta", "gamma delta", "eps zeta"], 600),
                   "x0": rng.normal(size=600) + 1.5 * lab})
tr, ev = Pool(td[:400], lab[:400], text_features=["txt"]), Pool(td[400:], lab[400:], text_features=["txt"])
for algorithm in ("RecursiveByShapValues", "RecursiveByLossFunctionChange"):
    m = CatBoostClassifier(iterations=40, random_seed=0, verbose=0, thread_count=N_JOBS, allow_writing_files=False)
    r = m.select_features(tr, eval_set=ev, features_for_select=["txt", "x0"], num_features_to_select=1, steps=1,
                          algorithm=algorithm, logging_level="Silent")
    imp = dict(zip(m.feature_names_, np.round(m.get_feature_importance(), 1)))
    print(f"{algorithm}: removed {r['eliminated_features_names']}, final model importance {imp}")

RecursiveByShapValues: removed ['txt'], final model importance {'txt': np.float64(14.2), 'x0': np.float64(85.8)}


RecursiveByLossFunctionChange: removed ['txt'], final model importance {'txt': np.float64(14.2), 'x0': np.float64(85.8)}


Both algorithms remove `txt`, and the final model still gives it a share of the importance.

## Groups of columns

With `grouping="ByTags"` the selection removes whole tags, defined on the `Pool`. Each tag also has a
cost, and the source divides a tag's score by it (ShapValues divides twice, source lines 297 and
425). The cell runs the selection with every cost at 1, then again with the cost of the city x
price tag at 10,000, which by the source should get that tag removed. `Pool` replaces the column
names in the tag lists it is given with column indices, in place, so the cell passes copies. By the
source (line 565) the zero-step bug above applies to tags too for PVC and LFC; that is not run here. Each pair is one tag, the additive columns another, and the noise two more; the selection keeps 4
of the 6 tags.

In [10]:
TAGS = {"additive": ADD_COLS, "city x price": ["city", "log_price"], "hour x distance": ["hour", "distance_km"],
        "u x v": ["u", "v"], "Gaussian noise": GAUSS_NOISE, "four-level noise": CAT_NOISE}
with timed("tags"):
    for price_cost in (1, 10_000):
        # copies: Pool replaces names with indices in place
        tags = {t: {"features": list(cols), "cost": price_cost if t == "city x price" else 1} for t, cols in TAGS.items()}
        tr = Pool(X, y, cat_features=CATS, feature_tags=tags)
        ev = Pool(X_eval, y_eval, cat_features=CATS, feature_tags=tags)
        r = model().select_features(tr, eval_set=ev, grouping="ByTags", features_tags_for_select=list(TAGS),
                                    num_features_tags_to_select=4, steps=2, algorithm="RecursiveByShapValues",
                                    train_final_model=False, logging_level="Silent")
        print(f"cost of city x price {price_cost}: kept {r['selected_features_tags']}; removed {r['eliminated_features_tags']}")

cost of city x price 1: kept ['hour x distance', 'city x price', 'u x v', 'additive']; removed ['Gaussian noise', 'four-level noise']


cost of city x price 10000: kept ['hour x distance', 'city x price', 'u x v', 'additive']; removed ['Gaussian noise', 'four-level noise']


The selection keeps the three pair tags and the additive tag and removes both noise tags, with the
city x price cost at 1 and at 10,000 alike: in this check the cost set through `Pool` has no effect.

In [11]:
print("Runtime per block, seconds:", {k: round(v) for k, v in TIMES.items()})
print(f"Total: {(time.perf_counter() - T0) / 60:.1f} min, N_JOBS = {N_JOBS}")

Runtime per block, seconds: {'call': 37, 'schedule': 173, 'algorithms': 74, 'loss graph': 61, 'eval_metric': 101, 'tags': 28}
Total: 8.5 min, N_JOBS = -1
